# Triage Pilot — Phase 0 Checkpoint 3: Eval Set Freeze

**Purpose:** Produce the frozen `test.jsonl` that every Phase 1+ metric will be measured against.

**Output artifacts** (written to Drive):
- `data/processed/train.jsonl`
- `data/processed/val.jsonl`
- `data/processed/test.jsonl`  ← frozen after this notebook commits its hash
- `eval/test_set_hash.txt`
- `eval/class_distribution.md`
- `eval/spot_check_samples.md`

**Freeze rule:** once `test_set_hash.txt` is committed to the repo, `test.jsonl` does not change. If labeling errors are found later, they go in `eval/known_errors.md` and corrections go into a future `test_v2.jsonl`.

## 1. Config

In [ ]:
import os

TRIAGE_ROOT = '/content/drive/MyDrive/setique/triage'
DATA_RAW_DIR = f'{TRIAGE_ROOT}/data/raw'
DATA_PROCESSED_DIR = f'{TRIAGE_ROOT}/data/processed'
EVAL_DIR = f'{TRIAGE_ROOT}/eval'

SEED = 42

# Split ratios
TRAIN_FRAC = 0.90
VAL_FRAC = 0.05
TEST_FRAC = 0.05

# Target test set size (upper bound — stratified sampling may produce slightly less)
TARGET_TEST_SIZE = 3000

# Minimum examples per class in test set — classes below this get merged or dropped
MIN_TEST_PER_CLASS = 30

# Datasets to pull (edit based on what you decide to include)
INCLUDE_BITEXT = True        # Bitext Customer Support — intent, category
INCLUDE_MASSIVE = True       # Multilingual intent — useful for SAM-Lang + cross-lingual SAM-Intent
INCLUDE_TWITTER = False      # Kaggle Twitter support — off by default (noisy, needs heavy filtering)

print(f'Seed: {SEED}')
print(f'Target test size: {TARGET_TEST_SIZE}')
print(f'Min per class in test: {MIN_TEST_PER_CLASS}')

## 2. Mount Drive + setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

# Deterministic seeding across the whole notebook
import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Install anything missing. Colab usually has datasets already.
!pip install -q datasets pandas scikit-learn

## 3. Label schemas

These are the locked label spaces. Classes below the minimum support threshold get merged into `other` or dropped, and the decisions are logged.

In [ ]:
LANG_CODES = {'en','es','fr','de','pt','it','nl','ja','zh','ko',
              'ar','ru','pl','tr','sv','hi','id','vi','th','other'}

INTENT_LABELS = {
    # Billing
    'billing_question','billing_dispute','refund_request','cancel_subscription',
    # Account
    'password_reset','account_access','update_profile','delete_account',
    # Product
    'how_to','feature_request','bug_report','compatibility_question',
    # Order
    'order_status','shipping_question','return_request','damaged_item',
    # Support tier
    'technical_issue','escalation_request','complaint','compliment',
    # Meta
    'spam','unclear','other',
}

PRIORITY_LABELS = {'P1','P2','P3','P4'}

ROUTE_LABELS = {
    'billing_team','technical_support','account_security',
    'returns_and_shipping','product_feedback','legal_compliance',
    'human_escalation','auto_close',
}

print(f'Languages:   {len(LANG_CODES)}')
print(f'Intents:     {len(INTENT_LABELS)}')
print(f'Priorities:  {len(PRIORITY_LABELS)}')
print(f'Routes:      {len(ROUTE_LABELS)}')

## 4. Load + normalize source datasets

Each loader returns a list of dicts with a common schema:
```
{
  'id': str,           # source-unique ID
  'text': str,         # the ticket text
  'lang': str | None,  # ISO 639-1 if known
  'intent': str | None,
  'priority': str | None,
  'route': str | None,
  'source': str,       # e.g. 'bitext', 'massive'
}
```

In [ ]:
from datasets import load_dataset
import hashlib

def _id(source, text):
    return f'{source}_{hashlib.md5(text.encode()).hexdigest()[:12]}'

def load_bitext():
    # Bitext taxonomy maps roughly to our INTENT_LABELS; remap in a separate step below
    ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
    out = []
    for row in ds:
        text = row.get('instruction') or row.get('utterance') or ''
        if not text.strip():
            continue
        out.append({
            'id': _id('bitext', text),
            'text': text.strip(),
            'lang': 'en',
            'intent': row.get('intent'),  # raw; gets remapped
            'priority': None,
            'route': None,
            'source': 'bitext',
        })
    return out

def load_massive():
    # MASSIVE: multilingual intent dataset from Amazon. Useful for SAM-Lang coverage.
    # Load a subset of locales for tractability — expand later if needed.
    locales = ['en-US','es-ES','fr-FR','de-DE','pt-PT','it-IT','ja-JP','zh-CN','ko-KR','ar-SA']
    out = []
    for loc in locales:
        try:
            ds = load_dataset('AmazonScience/massive', loc, split='train')
        except Exception as e:
            print(f'  skipped {loc}: {e}')
            continue
        lang2 = loc.split('-')[0]
        for row in ds:
            text = row.get('utt','').strip()
            if not text:
                continue
            out.append({
                'id': _id(f'massive_{lang2}', text),
                'text': text,
                'lang': lang2,
                'intent': row.get('intent_str') or row.get('intent'),
                'priority': None,
                'route': None,
                'source': 'massive',
            })
    return out

records = []
if INCLUDE_BITEXT:
    print('Loading Bitext...')
    records.extend(load_bitext())
    print(f'  cumulative records: {len(records)}')
if INCLUDE_MASSIVE:
    print('Loading MASSIVE...')
    records.extend(load_massive())
    print(f'  cumulative records: {len(records)}')

print(f'\nTotal raw records: {len(records)}')

## 5. Remap source intents → our intent taxonomy

Source datasets use their own intent labels. This cell maps them to `INTENT_LABELS`. Anything unmapped becomes `other`. **Review the mapping table — this is where labeling quality lives or dies.**

In [ ]:
# Starter mapping. Extend after inspecting source label distribution below.
# Leave a label unmapped to send it to 'other'.
INTENT_REMAP = {
    # Bitext
    'cancel_order': 'cancel_subscription',
    'change_order': 'order_status',
    'track_order': 'order_status',
    'get_refund': 'refund_request',
    'contact_customer_service': 'escalation_request',
    'contact_human_agent': 'escalation_request',
    'complaint': 'complaint',
    'recover_password': 'password_reset',
    'delete_account': 'delete_account',
    'edit_account': 'update_profile',
    'switch_account': 'account_access',
    'registration_problems': 'account_access',
    'place_order': 'how_to',
    'check_invoice': 'billing_question',
    'get_invoice': 'billing_question',
    'check_payment_methods': 'billing_question',
    'payment_issue': 'billing_dispute',
    'check_refund_policy': 'billing_question',
    'newsletter_subscription': 'update_profile',
    'review': 'compliment',
    'set_up_shipping_address': 'shipping_question',
    'check_cancellation_fee': 'billing_question',
    'delivery_options': 'shipping_question',
    'delivery_period': 'shipping_question',
    # MASSIVE (sampling — extend after inspection)
    'email_query': 'how_to',
    'email_sendemail': 'how_to',
    'calendar_set': 'how_to',
    'play_music': 'how_to',
    'iot_hue_lightchange': 'how_to',
    'general_quirky': 'unclear',
    'general_joke': 'spam',
}

# Inspect source label distribution to decide what else to add to the remap
from collections import Counter
raw_intent_counts = Counter(r['intent'] for r in records if r['intent'])
print('Top 40 raw source intent labels (extend INTENT_REMAP as needed):')
for label, cnt in raw_intent_counts.most_common(40):
    mapped = INTENT_REMAP.get(label, 'other')
    print(f'  {label:40s} n={cnt:6d}  ->  {mapped}')

In [ ]:
# Apply the remap
for r in records:
    raw = r.get('intent')
    if raw is None:
        r['intent'] = None
    elif raw in INTENT_REMAP:
        r['intent'] = INTENT_REMAP[raw]
    elif raw in INTENT_LABELS:
        r['intent'] = raw
    else:
        r['intent'] = 'other'

mapped_intent_counts = Counter(r['intent'] for r in records if r['intent'])
print('Mapped intent distribution:')
for label, cnt in mapped_intent_counts.most_common():
    print(f'  {label:30s} n={cnt}')

## 6. Heuristic priority + route labeling

Public data rarely has priority/route labels. Heuristic labels get generated from `(intent, text)` signals. These are noisy on purpose — they're training data. The **test set priority labels get reviewed by hand** before freeze.

In [ ]:
import re

P1_KEYWORDS = re.compile(r'\b(down|outage|urgent|emergency|security|breach|hacked|fraud|legal|lawsuit|lawyer)\b', re.I)
P2_KEYWORDS = re.compile(r'\b(broken|cannot|can\'t|locked out|not working|failed|error)\b', re.I)
P4_KEYWORDS = re.compile(r'\b(thanks|thank you|awesome|love|amazing|great job)\b', re.I)

def heuristic_priority(intent, text):
    if intent in ('escalation_request',) or P1_KEYWORDS.search(text or ''):
        return 'P1'
    if intent in ('billing_dispute','bug_report','account_access','damaged_item') or P2_KEYWORDS.search(text or ''):
        return 'P2'
    if intent in ('compliment',) or P4_KEYWORDS.search(text or ''):
        return 'P4'
    return 'P3'

ROUTE_BY_INTENT = {
    'billing_question':'billing_team','billing_dispute':'billing_team',
    'refund_request':'billing_team','cancel_subscription':'billing_team',
    'password_reset':'account_security','account_access':'account_security',
    'update_profile':'account_security','delete_account':'account_security',
    'how_to':'technical_support','bug_report':'technical_support',
    'compatibility_question':'technical_support','technical_issue':'technical_support',
    'feature_request':'product_feedback','compliment':'product_feedback',
    'order_status':'returns_and_shipping','shipping_question':'returns_and_shipping',
    'return_request':'returns_and_shipping','damaged_item':'returns_and_shipping',
    'escalation_request':'human_escalation','complaint':'human_escalation',
    'spam':'auto_close','unclear':'human_escalation','other':'human_escalation',
}

for r in records:
    if r['priority'] is None:
        r['priority'] = heuristic_priority(r['intent'], r['text'])
    if r['route'] is None:
        r['route'] = ROUTE_BY_INTENT.get(r['intent'] or 'other', 'human_escalation')

print('Priority distribution:', Counter(r['priority'] for r in records))
print('Route distribution:   ', Counter(r['route'] for r in records))

## 7. Dedup on normalized text

In [ ]:
def normalize(text):
    return re.sub(r'\s+', ' ', (text or '').lower()).strip()

seen = set()
deduped = []
for r in records:
    key = normalize(r['text'])
    if len(key) < 5:
        continue
    if key in seen:
        continue
    seen.add(key)
    deduped.append(r)

print(f'Before dedup: {len(records)}')
print(f'After dedup:  {len(deduped)}')
records = deduped

## 8. Drop low-support intent classes

Anything below `MIN_TEST_PER_CLASS / TEST_FRAC` in total support can't produce a stable test slice. Merge to `other`.

In [ ]:
MIN_TOTAL_PER_CLASS = int(MIN_TEST_PER_CLASS / TEST_FRAC)
print(f'Minimum total examples for a class to survive: {MIN_TOTAL_PER_CLASS}')

intent_counts = Counter(r['intent'] for r in records)
demoted = [lbl for lbl, n in intent_counts.items() if n < MIN_TOTAL_PER_CLASS and lbl != 'other']

print('Demoted to other:')
for lbl in demoted:
    print(f'  {lbl:30s} n={intent_counts[lbl]}')

for r in records:
    if r['intent'] in demoted:
        r['intent'] = 'other'

print('\nFinal intent distribution:')
for label, cnt in Counter(r['intent'] for r in records).most_common():
    print(f'  {label:30s} n={cnt}')

## 9. Stratified split

Stratify on `intent` (the hardest-to-balance label). `lang`, `priority`, `route` distributions get audited after the fact.

In [ ]:
from sklearn.model_selection import train_test_split

intents = [r['intent'] for r in records]

# First split: train vs (val+test)
train_records, holdout_records = train_test_split(
    records, test_size=(VAL_FRAC + TEST_FRAC),
    stratify=intents, random_state=SEED,
)

# Second split: val vs test, stratified again
holdout_intents = [r['intent'] for r in holdout_records]
val_size = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
val_records, test_records = train_test_split(
    holdout_records, test_size=(1 - val_size),
    stratify=holdout_intents, random_state=SEED,
)

# Optional downsample of test to TARGET_TEST_SIZE, stratified
if len(test_records) > TARGET_TEST_SIZE:
    test_intents = [r['intent'] for r in test_records]
    test_records, _ = train_test_split(
        test_records, train_size=TARGET_TEST_SIZE,
        stratify=test_intents, random_state=SEED,
    )

print(f'Train: {len(train_records)}')
print(f'Val:   {len(val_records)}')
print(f'Test:  {len(test_records)}')

## 10. Write splits to Drive

In [ ]:
import json

def write_jsonl(records, path):
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

train_path = f'{DATA_PROCESSED_DIR}/train.jsonl'
val_path = f'{DATA_PROCESSED_DIR}/val.jsonl'
test_path = f'{DATA_PROCESSED_DIR}/test.jsonl'

write_jsonl(train_records, train_path)
write_jsonl(val_records, val_path)
write_jsonl(test_records, test_path)

for p in [train_path, val_path, test_path]:
    size_mb = os.path.getsize(p) / (1024**2)
    print(f'{p}  ({size_mb:.2f} MB)')

## 11. Hash + freeze the test set

In [ ]:
import hashlib

def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

test_hash = sha256_of_file(test_path)
hash_path = f'{EVAL_DIR}/test_set_hash.txt'
with open(hash_path, 'w') as f:
    f.write(f'{test_hash}  test.jsonl\n')

print(f'Test set SHA256: {test_hash}')
print(f'Hash written to: {hash_path}')
print('\n*** Commit test_set_hash.txt to the repo. After this, test.jsonl is frozen. ***')

## 12. Class distribution audit

In [ ]:
def distribution_table(recs, field):
    counts = Counter(r[field] for r in recs)
    total = sum(counts.values())
    lines = [f'| {field} | count | pct |', '|---|---|---|']
    for label, cnt in counts.most_common():
        lines.append(f'| {label} | {cnt} | {100*cnt/total:.2f}% |')
    return '\n'.join(lines)

md = ['# Eval set class distribution\n',
      f'Seed: {SEED}. Test hash: `{test_hash[:16]}...`\n',
      f'Train/Val/Test: {len(train_records)} / {len(val_records)} / {len(test_records)}\n']

for split_name, recs in [('Test','test'), ('Val','val'), ('Train','train')]:
    r = {'test': test_records, 'val': val_records, 'train': train_records}[recs]
    md.append(f'\n## {split_name} — intent\n')
    md.append(distribution_table(r, 'intent'))
    md.append(f'\n\n## {split_name} — lang\n')
    md.append(distribution_table(r, 'lang'))
    md.append(f'\n\n## {split_name} — priority\n')
    md.append(distribution_table(r, 'priority'))
    md.append(f'\n\n## {split_name} — route\n')
    md.append(distribution_table(r, 'route'))

# Flag low-support test classes
test_intent_counts = Counter(r['intent'] for r in test_records)
low_support = [(lbl, n) for lbl, n in test_intent_counts.items() if n < MIN_TEST_PER_CLASS]
if low_support:
    md.append('\n\n## ⚠️ Low-support classes in test set\n')
    md.append(f'These classes have fewer than {MIN_TEST_PER_CLASS} examples in test — metrics will be unstable:\n')
    for lbl, n in low_support:
        md.append(f'- `{lbl}`: {n}')

dist_path = f'{EVAL_DIR}/class_distribution.md'
with open(dist_path, 'w') as f:
    f.write('\n'.join(md))

print(f'Class distribution written to: {dist_path}')
if low_support:
    print('\n⚠️  Low-support classes detected in test set — review before proceeding.')

## 13. Spot-check samples

One example per intent class. Skim these manually and flag any mislabelings in `eval/known_errors.md`. **Do not fix them in test.jsonl** — the frozen test set includes its labeling errors. Fixes go in test_v2 later.

In [ ]:
rng = random.Random(SEED)
by_intent = {}
for r in test_records:
    by_intent.setdefault(r['intent'], []).append(r)

md = ['# Eval set spot-check samples\n',
      'One random test example per intent class. Review for label correctness.\n',
      'Flag any mislabels in `eval/known_errors.md` — do NOT edit test.jsonl.\n']

for intent in sorted(by_intent.keys()):
    sample = rng.choice(by_intent[intent])
    md.append(f'\n## {intent}\n')
    md.append(f'- **id:** `{sample["id"]}`')
    md.append(f'- **lang:** {sample["lang"]}')
    md.append(f'- **priority:** {sample["priority"]}')
    md.append(f'- **route:** {sample["route"]}')
    md.append(f'- **text:** {sample["text"][:500]}')

spot_path = f'{EVAL_DIR}/spot_check_samples.md'
with open(spot_path, 'w') as f:
    f.write('\n'.join(md))

print(f'Spot-check samples written to: {spot_path}')
print(f'{len(by_intent)} classes sampled')

## 14. Checkpoint 3 verdict

Before marking green:

- [ ] No low-support warnings in the class distribution (or they're accepted and documented)
- [ ] Spot-check samples reviewed — obvious mislabelings caught
- [ ] `test_set_hash.txt` committed to the repo
- [ ] `class_distribution.md` and `spot_check_samples.md` committed to the repo
- [ ] `data/processed/*.jsonl` lives in Drive, excluded from git via `.gitignore`

Once all green, proceed to Checkpoint 4 (baselines).